# E2 Qwen3-8B QLoRA private Kaggle runner
Attach `e2_kaggle_source.zip`, the restricted corpus, and the private output from the completed 3-epoch E2 run, then select a GPU runtime. The notebook restores the latest resumable Trainer checkpoint and continues to a total of 5 epochs (exactly 2 additional epochs).

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

EXPERIMENT_ID = "e2_qwen3_8b_qlora_vi_zh_v1"
ADDITIONAL_EPOCHS = 2
archives = list(Path("/kaggle/input").rglob("e2_kaggle_source.zip"))
if len(archives) != 1:
    raise RuntimeError(f"Expected one attached e2_kaggle_source.zip, found {archives}")
REPO_DIR = Path("/kaggle/working/nlp-proj")
REPO_DIR.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(archives[0]), str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = str(REPO_DIR / "code")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def run_command(*args):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], cwd=REPO_DIR, env=os.environ.copy(), check=True)

saved_manifests = [
    path for path in Path("/kaggle/input").rglob("run_manifest.json")
    if path.parent.name == EXPERIMENT_ID
]
if len(saved_manifests) != 1:
    raise RuntimeError(f"Expected exactly one saved E2 run to resume, found {saved_manifests}")
manifest_source = saved_manifests[0]
prior_work = manifest_source.parent
prior_repo = manifest_source.parents[2]
manifest = json.loads(manifest_source.read_text(encoding="utf-8"))
prior_checkpoints = []
for path in (prior_work / "trainer").glob("checkpoint-*"):
    try:
        step = int(path.name.rsplit("-", 1)[1])
    except ValueError:
        continue
    if path.is_dir():
        prior_checkpoints.append((step, path))
if not prior_checkpoints:
    raise RuntimeError("The attached E2 output has no work/.../trainer/checkpoint-* directory; a final adapter alone cannot restore optimizer and scheduler state")
resume_source = max(prior_checkpoints)[1]
resume_state = json.loads((resume_source / "trainer_state.json").read_text(encoding="utf-8"))
START_EPOCH = float(resume_state["epoch"])
TARGET_EPOCHS = START_EPOCH + ADDITIONAL_EPOCHS
destination_work = REPO_DIR / "work" / EXPERIMENT_ID
shutil.copytree(prior_work, destination_work, dirs_exist_ok=True)
prior_train_log = destination_work / "train.log"
archived_train_log = destination_work / "train.initial_3_epochs.log"
if prior_train_log.exists():
    prior_train_log.replace(archived_train_log)
source_adapter = prior_repo / "checkpoint" / EXPERIMENT_ID / "adapter"
destination_adapter = REPO_DIR / "checkpoint" / EXPERIMENT_ID / "adapter"
if source_adapter.is_dir():
    shutil.copytree(source_adapter, destination_adapter, dirs_exist_ok=True)
print(f"Recovered {resume_source.name} at epoch {START_EPOCH:g}; training target is epoch {TARGET_EPOCHS:g}.")
print(REPO_DIR)


In [ ]:
from importlib.metadata import version
expected_versions = {}
for line in (REPO_DIR / "requirements-llm.lock.txt").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        name, expected = line.split("==", 1)
        expected_versions[name] = expected
for name, saved_version in manifest["packages"].items():
    if saved_version:
        expected_versions[name] = saved_version
install_versions = {name: saved for name, saved in expected_versions.items() if name != "torch"}
install_args = [f"{name}=={saved}" for name, saved in sorted(install_versions.items())]
run_command(sys.executable, "-m", "pip", "install", "--no-cache-dir", *install_args)
actual_versions = {name: version(name) for name in expected_versions}
assert actual_versions == expected_versions, (expected_versions, actual_versions)
print("Verified direct packages:", actual_versions)

import torch
assert torch.cuda.is_available(), "Select a Kaggle GPU runtime before running E2"
props = torch.cuda.get_device_properties(0)
assert props.total_memory >= 14 * 1024**3, f"E2 requires at least 14 GiB VRAM, found {props.total_memory / 1024**3:.1f}"
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), f"{props.total_memory / 1024**3:.1f} GiB")

adapter_dir = REPO_DIR / "checkpoint" / EXPERIMENT_ID / "adapter"
manifest_path = REPO_DIR / "work" / EXPERIMENT_ID / "run_manifest.json"
resume_checkpoint = REPO_DIR / "work" / EXPERIMENT_ID / "trainer" / resume_source.name
required = [resume_checkpoint / name for name in ("adapter_config.json", "trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth")]
weights = [resume_checkpoint / "adapter_model.safetensors", resume_checkpoint / "adapter_model.bin"]
missing = [str(path) for path in required if not path.exists()]
if not any(path.exists() for path in weights):
    missing.append(str(weights))
assert not missing, missing
resume_config = dict(manifest["config"])
resume_config["training"] = dict(resume_config["training"])
resume_config["training"]["epochs"] = TARGET_EPOCHS
resume_config["resume"] = True
CONFIG_PATH = REPO_DIR / "configs" / "e2_qwen3_qlora_resume.yaml"
CONFIG_PATH.write_text(json.dumps(resume_config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Verified resumable state and wrote {CONFIG_PATH.name} with epochs={TARGET_EPOCHS:g}, resume=true.")


In [ ]:
run_command(sys.executable, "-m", "pytest", "-q", "tests")
run_command(sys.executable, "-m", "mt_pipeline", "train", "--config", CONFIG_PATH)


Validation is generated and scored before selection is frozen. Test generation is refused until the freeze artifact exists. Kaggle may report dependency conflicts for unrelated preinstalled packages, and PyTorch 2.10 may emit an AccumulateGrad stream warning; neither is a pipeline failure when the pinned-version assertions, tests, and subprocess exit checks pass.

In [ ]:
run_command(sys.executable, "-m", "mt_pipeline", "predict", "--config", CONFIG_PATH, "--split", "val")
run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", "predictions/e2_qwen3_8b_qlora_vi_zh_v1.val.jsonl", "--output", "metrics/e2_qwen3_8b_qlora_vi_zh_v1.val.json")
run_command(sys.executable, "-m", "mt_pipeline", "freeze-selection", "--config", CONFIG_PATH, "--validation-predictions", "predictions/e2_qwen3_8b_qlora_vi_zh_v1.val.jsonl", "--validation-metrics", "metrics/e2_qwen3_8b_qlora_vi_zh_v1.val.json")


In [ ]:
run_command(sys.executable, "-m", "mt_pipeline", "predict", "--config", CONFIG_PATH, "--split", "test")
run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", "predictions/e2_qwen3_8b_qlora_vi_zh_v1.test.jsonl", "--output", "metrics/e2_qwen3_8b_qlora_vi_zh_v1.test.json")
run_command(sys.executable, "-m", "mt_pipeline", "project-status")


In [ ]:
work_dir = REPO_DIR / "work" / EXPERIMENT_ID
trainer_checkpoints = []
for path in (work_dir / "trainer").glob("checkpoint-*"):
    try:
        trainer_checkpoints.append((int(path.name.rsplit("-", 1)[1]), path))
    except ValueError:
        pass
assert trainer_checkpoints, "Training produced no resumable checkpoint"
latest_checkpoint = max(trainer_checkpoints)[1]
final_state = json.loads((work_dir / "trainer" / "trainer_state.json").read_text(encoding="utf-8"))
assert float(final_state["epoch"]) == TARGET_EPOCHS, final_state["epoch"]
best_checkpoint = work_dir / "trainer" / Path(final_state["best_model_checkpoint"]).name
result_paths = [
    adapter_dir,
    latest_checkpoint,
    work_dir / "run_manifest.json",
    work_dir / "training_history.json",
    work_dir / "train.log",
    work_dir / "train.initial_3_epochs.log",
    work_dir / "trainer" / "trainer_state.json",
    work_dir / "selection_frozen.json",
    REPO_DIR / "predictions" / f"{EXPERIMENT_ID}.val.jsonl",
    REPO_DIR / "predictions" / f"{EXPERIMENT_ID}.test.jsonl",
    REPO_DIR / "metrics" / f"{EXPERIMENT_ID}.val.json",
    REPO_DIR / "metrics" / f"{EXPERIMENT_ID}.test.json",
]
if best_checkpoint != latest_checkpoint:
    result_paths.append(best_checkpoint)
missing = [str(path) for path in result_paths if not path.exists()]
assert not missing, missing
stage = Path("/kaggle/working/e2_qwen3_qlora_results")
if stage.exists():
    shutil.rmtree(stage)
for source in result_paths:
    destination = stage / source.relative_to(REPO_DIR)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination)
    else:
        shutil.copy2(source, destination)
archive = shutil.make_archive(str(stage), "zip", root_dir=stage)
print(f"Download {archive}; it contains the final adapter, the latest resumable checkpoint, the best checkpoint, and E2 evaluation artifacts.")
